# Historique Banlist TCG — Scraping Yugipedia (TOK-8)

Scrape les 81 banlists TCG Advanced Format depuis yugipedia via l'API MediaWiki.

**Table produite :** `banlist_history`
- `list_name` : ex. "May 2026 Lists (TCG)"
- `effective_date` / `end_date` : période d'application
- `card_name` / `status` : Forbidden / Limited / Semi-Limited

**Résultat :** 11 890 lignes, 81 banlists, 522 cartes uniques, période 2002 → 2026.

In [ ]:
import requests, re, sqlite3, time
import pandas as pd
from datetime import datetime

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}

r = requests.get(
    'https://yugipedia.com/api.php?action=query&list=categorymembers'
    '&cmtitle=Category:TCG_Advanced_Format_Forbidden_%26_Limited_Lists&cmlimit=200&format=json',
    headers=headers, timeout=15
)
members = r.json().get('query', {}).get('categorymembers', [])
print(f'{len(members)} banlists TCG Advanced Format trouvées')

In [ ]:
def parse_date(s):
    s = s.strip()
    if not s: return None
    for fmt in ('%B %d, %Y', '%Y-%m-%d', '%d %B %Y'):
        try: return datetime.strptime(s, fmt).strftime('%Y-%m-%d')
        except: pass
    return None

def parse_banlist_wikitext(title, text):
    rows = []
    start_m = re.search(r'\|\s*start_date\s*=\s*(.+)', text)
    end_m   = re.search(r'\|\s*end_date\s*=\s*(.+)', text)
    effective_date = parse_date(start_m.group(1)) if start_m else None
    end_date       = parse_date(end_m.group(1))   if end_m   else None
    for status_key, status_label in [('forbidden','Forbidden'), ('limited','Limited'), ('semi_limited','Semi-Limited')]:
        pat = (r'\|\s*' + status_key +
               r'\s*=\s*(.*?)(?=\|\s*(?:forbidden|limited|semi_limited|unlimited'
               r'|notes|prev|next|start_date|end_date|medium|format)\s*=|\}\})')
        m = re.search(pat, text, re.DOTALL | re.IGNORECASE)
        if not m: continue
        for line in m.group(1).split('\n'):
            card = re.sub(r'//.*$', '', line)
            card = re.sub(r'\[\[|\]\]', '', card).strip()
            if not card or card.startswith('|') or card.startswith('*') or card.startswith('{'): continue
            rows.append({'list_name': title, 'effective_date': effective_date,
                         'end_date': end_date, 'card_name': card, 'status': status_label})
    return rows

In [ ]:
all_rows = []
errors   = []

for i, m in enumerate(members):
    title = m['title']
    for attempt in range(3):
        try:
            r2 = requests.get(
                'https://yugipedia.com/api.php?action=parse&page=' +
                requests.utils.quote(title) + '&prop=wikitext&format=json',
                headers=headers, timeout=30
            )
            wikitext = r2.json().get('parse', {}).get('wikitext', {}).get('*', '')
            rows = parse_banlist_wikitext(title, wikitext)
            all_rows.extend(rows)
            break
        except Exception as e:
            if attempt == 2: errors.append(f'{title}: {e}')
            else: time.sleep(2)
    time.sleep(0.4)
    if (i+1) % 10 == 0:
        print(f'[{i+1}/{len(members)}] {len(all_rows)} entrées...')

print(f'\nTotal : {len(all_rows)} entrées, {len(errors)} erreurs')
if errors: print('Erreurs:', errors[:5])

In [ ]:
df = pd.DataFrame(all_rows)
print(f'Distrib statuts:\n{df["status"].value_counts()}')
print(f'Dates: {df["effective_date"].dropna().min()} → {df["effective_date"].dropna().max()}')
print(f'Cartes uniques: {df["card_name"].nunique()}')

con = sqlite3.connect('../data/yugioh.db')
con.execute('DROP TABLE IF EXISTS banlist_history')
con.execute('''CREATE TABLE banlist_history (
    list_name TEXT, effective_date TEXT, end_date TEXT,
    card_name TEXT, status TEXT
)''')
df.to_sql('banlist_history', con, if_exists='append', index=False)
con.execute('CREATE INDEX idx_banlist_card ON banlist_history(card_name)')
con.execute('CREATE INDEX idx_banlist_date ON banlist_history(effective_date)')
con.commit()
con.close()
print(f'\n✓ {len(df):,} lignes sauvegardées dans banlist_history')